# 02 - Nettoyage des données DPE

**Objectif** : préparer des données propres pour les départements 59, 69 et 75.
Chaque département est traité séparément, avec les mêmes règles appliquées à tous.

## 1. Chargement
Les trois fichiers `processed` (neufs + existants déjà fusionnés par département) sont chargés dans un dictionnaire : une entrée par département.

In [20]:
import sys
print(sys.executable)

c:\Users\albar\Documents\GitHub\SISE-CONSO-ANALYSIS\.venv\Scripts\python.exe


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 50)

DOSSIER = "../data/processed/2026-10-08_214716/"
DEPARTEMENTS = ["59", "69", "75"]

# Un dictionnaire : une entrée par département
dfs = {d: pd.read_parquet(DOSSIER + f"{d}.parquet") for d in DEPARTEMENTS}

# Une boucle qui affiche la taille de chacun
for d, df in dfs.items():
    print(d, df.shape)

59 (20000, 224)
69 (20000, 224)
75 (18644, 221)


## 2. Comparaison des trois départements
Paris compte moins de lignes et de colonnes que les deux autres départements. On vérifie la répartition neufs / existants et les colonnes manquantes.

In [2]:
# Répartition neufs / existants par département
for d, df in dfs.items():
    print(d, df["origine"].value_counts().to_dict())

# Colonnes qui ne sont pas présentes dans tous les départements
toutes = set().union(*[set(df.columns) for df in dfs.values()])
for d, df in dfs.items():
    absentes = sorted(toutes - set(df.columns))
    print(f"\n{d} : {len(absentes)} colonne(s) absente(s) -> {absentes}")

59 {'dpe_existants': 10000, 'dpe_neufs': 10000}
69 {'dpe_existants': 10000, 'dpe_neufs': 10000}
75 {'dpe_existants': 10000, 'dpe_neufs': 8644}

59 : 0 colonne(s) absente(s) -> []

69 : 0 colonne(s) absente(s) -> []

75 : 3 colonne(s) absente(s) -> ['facteur_couverture_solaire_installation_chauffage_n1', 'facteur_couverture_solaire_saisi_installation_chauffage_n1', 'numero_rpls_logement']


**Observation (étape 2)**

- **Paris** a 8 644 logements neufs, contre 10 000 pour Lille et Lyon. Pour Paris, on a donc sans doute tous les DPE neufs.
- Dans nos fichiers, il y a environ **moitié neufs, moitié existants**. Ce n'est pas la réalité : c'est parce qu'on a récupéré au maximum 10 000 lignes de chaque type. Il ne faut donc pas en tirer de conclusion sur le département.
- Paris n'a pas 3 colonnes très rares (sur le solaire et un numéro RPLS). Elles ne servent pas dans le projet, ce n'est pas un problème.

## 3. Vue d'ensemble par département

Pour chaque département, on regarde : les doublons, la période couverte et la répartition des étiquettes DPE.

In [3]:
for d, df in dfs.items():
    print(f"===== Département {d} =====")
    print("Doublons sur numero_dpe :", df["numero_dpe"].duplicated().sum())

    dates = pd.to_datetime(df["date_reception_dpe"], errors="coerce")
    print("Période :", dates.min().date(), "->", dates.max().date())

    print("Étiquettes :")
    print(pd.crosstab(df["origine"], df["etiquette_dpe"]))
    print()

===== Département 59 =====
Doublons sur numero_dpe : 0
Période : 2021-07-01 -> 2026-02-09
Étiquettes :
etiquette_dpe     A     B     C     D     E    F    G
origine                                              
dpe_existants    74   236  2879  3640  1989  752  430
dpe_neufs      1883  1988  6082    46     0    0    1

===== Département 69 =====
Doublons sur numero_dpe : 0
Période : 2021-07-01 -> 2026-02-09
Étiquettes :
etiquette_dpe     A     B     C     D     E    F    G
origine                                              
dpe_existants    64   324  3804  3194  1646  576  392
dpe_neufs      1781  2087  6072    46     2    0   12

===== Département 75 =====
Doublons sur numero_dpe : 0
Période : 2021-07-06 -> 2026-09-14
Étiquettes :
etiquette_dpe     A     B     C     D     E     F    G
origine                                               
dpe_existants     1    58  1639  2765  3642  1224  671
dpe_neufs      3021  2333  3252    37     1     0    0



**Observations (étape 3)**
- Aucun doublon dans les trois départements.
- Les logements neufs sont presque tous en A, B ou C.
- Les logements existants de Paris sont plus souvent en E que ceux de Lille et de Lyon.
- Les neufs en G sont presque tous à Lyon (12), très peu à Lille (1) et aucun à Paris.
- Lille et Lyon s'arrêtent au 09/02/2026, Paris va jusqu'au 14/09/2026 : à vérifier, nos données ne couvrent peut-être pas toute la période.

## 4. Répartition des DPE dans le temps
On regarde combien de DPE il y a par année et par type, pour voir si l'échantillon couvre bien toute la période.

In [4]:
for d, df in dfs.items():
    dates = pd.to_datetime(df["date_reception_dpe"], errors="coerce")
    print(f"===== Département {d} =====")
    print(pd.crosstab(dates.dt.year, df["origine"]))
    print("Dernière date par type :")
    print(dates.groupby(df["origine"]).max().dt.date)
    print()

===== Département 59 =====
origine             dpe_existants  dpe_neufs
date_reception_dpe                          
2021                         4793        962
2022                           61       2508
2023                           47       2338
2024                          328       2561
2025                          184       1256
2026                         4587        375
Dernière date par type :
origine
dpe_existants    2026-02-09
dpe_neufs        2026-02-09
Name: date_reception_dpe, dtype: object

===== Département 69 =====
origine             dpe_existants  dpe_neufs
date_reception_dpe                          
2021                         4978       1692
2022                           43       2529
2023                          134       2027
2024                          301       2151
2025                          400       1050
2026                         4144        551
Dernière date par type :
origine
dpe_existants    2026-02-09
dpe_neufs        2026-02-09
Name: d

**Observations (étape 4)**
- Existants de Lille et Lyon : environ la moitié des DPE sont de 2021, l'autre moitié de 2026, et presque aucun entre les deux.
- Existants de Paris : presque tous de 2026.
- Lille et Lyon s'arrêtent au 09/02/2026 : il manque les mois suivants.
- Conséquence : l'échantillon n'est pas représentatif dans le temps. On ne peut pas étudier l'évolution dans le temps avec ces données.

## 5. Les DPE de 2021 et de 2026 sont-ils comparables ?
Les règles de calcul du DPE ont pu changer entre les deux périodes. On compare, pour les logements existants, la répartition des étiquettes et la consommation médiane par année.

In [5]:
for d, df in dfs.items():
    ex = df[df["origine"] == "dpe_existants"].copy()
    ex["annee"] = pd.to_datetime(ex["date_reception_dpe"], errors="coerce").dt.year
    print(f"===== Département {d} (existants) =====")

    # Part de chaque étiquette par année
    print(pd.crosstab(ex["annee"], ex["etiquette_dpe"], normalize="index").round(2))

    # Consommation médiane par année, pour les 3 énergies de chauffage les plus fréquentes
    top = ex["type_energie_principale_chauffage"].value_counts().head(3).index
    print(ex[ex["type_energie_principale_chauffage"].isin(top)]
          .pivot_table(index="annee", columns="type_energie_principale_chauffage",
                       values="conso_5_usages_par_m2_ep", aggfunc="median").round(0))
    print()

===== Département 59 (existants) =====
etiquette_dpe     A     B     C     D     E     F     G
annee                                                  
2021           0.00  0.01  0.24  0.35  0.24  0.10  0.06
2022           0.03  0.05  0.52  0.20  0.11  0.05  0.03
2023           0.00  0.00  0.38  0.40  0.17  0.04  0.00
2024           0.00  0.01  0.60  0.24  0.13  0.02  0.00
2025           0.01  0.14  0.42  0.21  0.10  0.11  0.01
2026           0.01  0.03  0.31  0.40  0.16  0.06  0.03
type_energie_principale_chauffage  Gaz naturel  Réseau de Chauffage urbain  \
annee                                                                        
2021                                     216.0                       243.0   
2022                                     134.0                       431.0   
2023                                     135.0                       333.0   
2024                                     139.0                       239.0   
2025                                     175.

**Observations (étape 5)**
- Pour les logements existants, les étiquettes et la consommation médiane de 2021 et 2026 sont comparables dans chaque département : on garde toutes les années.
- Les années 2022 à 2025 ont trop peu de lignes (moins de 400) pour être interprétées.
- Paris est différent de Lille et de Lyon : plus de logements en E, F et G, et consommation électrique plus élevée. C'est une vraie différence du parc, pas une erreur.
- À mentionner dans le rapport : nos données sont concentrées sur 2021 et 2026, elles ne permettent pas d'étudier l'évolution dans le temps car cela dessinerait deux pics sur 2021 et 2026.

## 6. Consommation et valeurs aberrantes
Variable : `conso_5_usages_par_m2_ep` (kWh/m²/an). On cherche les valeurs impossibles ou extrêmes pour fixer un seuil, en regardant chaque département.

In [6]:
col = "conso_5_usages_par_m2_ep"

for d, df in dfs.items():
    print(f"===== Département {d} =====")
    print(df[col].quantile([0.01, 0.5, 0.9, 0.99, 0.999, 1]).round(1))
    print("Valeurs <= 0  :", (df[col] <= 0).sum())
    print("Valeurs > 800 :", (df[col] > 800).sum())
    print("5 plus fortes valeurs :")
    print(df.nlargest(5, col)[["etiquette_dpe", col, "surface_habitable_logement", "origine"]])
    print()

===== Département 59 =====
0.010      30.5
0.500      94.0
0.900     285.0
0.990     483.7
0.999     731.5
1.000    2014.3
Name: conso_5_usages_par_m2_ep, dtype: float64
Valeurs <= 0  : 0
Valeurs > 800 : 17
5 plus fortes valeurs :
     etiquette_dpe  conso_5_usages_par_m2_ep  surface_habitable_logement  \
5813             G                    2014.3                        82.1   
1124             G                    1869.0                        34.4   
1215             G                    1853.3                        53.6   
7309             G                    1751.0                        12.0   
2061             G                    1726.5                        41.7   

            origine  
5813  dpe_existants  
1124  dpe_existants  
1215  dpe_existants  
7309  dpe_existants  
2061  dpe_existants  

===== Département 69 =====
0.010      34.3
0.500      98.2
0.900     265.0
0.990     476.0
0.999     758.7
1.000    1811.1
Name: conso_5_usages_par_m2_ep, dtype: float64
Valeurs <

**Observations (étape 6)**
- Aucune consommation nulle ou négative.
- Peu de valeurs au-dessus de 800 kWh/m²/an : 17 à Lille, 14 à Lyon, 42 à Paris (environ 0,12 % des lignes).
- Paris a des consommations plus élevées (médiane de 148 contre environ 95 ailleurs), ce qui est cohérent avec son parc ancien et ses petites surfaces.
- Deux cas parmi les valeurs extrêmes : des très petites surfaces (8 à 20 m²), où une forte consommation par m² est possible, et des surfaces normales (environ 80 m²) avec plus de 1 800 kWh/m²/an, très probablement des erreurs de saisie.

## 7. Choix du seuil maximum de consommation
On compare plusieurs seuils : combien de lignes supprimerait chacun, et à quelles surfaces correspondent les valeurs extrêmes.

In [7]:
col = "conso_5_usages_par_m2_ep"
surf = "surface_habitable_logement"

# 1. Combien de lignes seraient supprimées selon le seuil ?
lignes = []
for d, df in dfs.items():
    for seuil in [600, 800, 1000, 1200]:
        n = (df[col] > seuil).sum()
        lignes.append({"departement": d, "seuil": seuil,
                       "lignes_supprimees": n,
                       "part_%": round(n / len(df) * 100, 2)})
pd.DataFrame(lignes).pivot(index="seuil", columns="departement",
                           values=["lignes_supprimees", "part_%"])

lignes_supprimees              part_%            
departement                59    69     75     59    69    75
seuil                                                        
600                      70.0  77.0  237.0   0.35  0.38  1.27
800                      17.0  14.0   42.0   0.08  0.07  0.23
1000                      8.0   3.0    7.0   0.04  0.02  0.04
1200                      7.0   3.0    2.0   0.03  0.02  0.01

In [8]:
# 2. À quelles surfaces correspondent les valeurs > 800 ?
for d, df in dfs.items():
    extremes = df[df[col] > 800]
    print(f"===== Département {d} : {len(extremes)} valeurs > 800 =====")
    print(pd.cut(extremes[surf], [0, 9, 20, 50, 1000]).value_counts().sort_index())
    print("Surface manquante :", extremes[surf].isna().sum())
    print()

===== Département 59 : 17 valeurs > 800 =====
surface_habitable_logement
(0, 9]        0
(9, 20]       4
(20, 50]      6
(50, 1000]    6
Name: count, dtype: int64
Surface manquante : 1

===== Département 69 : 14 valeurs > 800 =====
surface_habitable_logement
(0, 9]        2
(9, 20]       5
(20, 50]      3
(50, 1000]    3
Name: count, dtype: int64
Surface manquante : 1

===== Département 75 : 42 valeurs > 800 =====
surface_habitable_logement
(0, 9]        16
(9, 20]       21
(20, 50]       4
(50, 1000]     1
Name: count, dtype: int64
Surface manquante : 0



**Observations (étape 7)**
- Un seuil à 600 kWh/m²/an est trop sévère : il supprimerait de vraies passoires thermiques (la classe G commence vers 420 kWh/m²/an).
- Au-dessus de 1 000, il ne reste que quelques valeurs isolées (8 à Lille, 3 à Lyon, 7 à Paris), très probablement des erreurs de saisie.
- À Paris, presque toutes les consommations supérieures à 800 concernent des logements de moins de 20 m², plausibles pour de très petits logements.
- À Lille et à Lyon, quelques valeurs supérieures à 800 concernent des logements de plus de 50 m² : peu crédibles.

**Décision proposée** : seuil maximum de consommation de **1 000 kWh/m²/an**. Il supprime 18 lignes sur 58 644 (0,03 %) et conserve les petits logements parisiens plausibles.

## 8. Surface et type de bâtiment
La règle sur la surface est liée à celle sur la consommation. On regarde les surfaces trop petites ou trop grandes, les surfaces manquantes, et le type de bâtiment concerné.

In [9]:
surf = "surface_habitable_logement"

for d, df in dfs.items():
    print(f"===== Département {d} =====")
    print(df[surf].describe().round(1))
    print("Surface manquante :", df[surf].isna().sum())
    print("Surface < 9 m²    :", (df[surf] < 9).sum())
    print("Surface > 300 m²  :", (df[surf] > 300).sum())
    print("Surface > 500 m²  :", (df[surf] > 500).sum())
    print("Hauteur sous plafond :")
    print(df["hauteur_sous_plafond"].describe().round(2))
    print("Type de bâtiment, quand la surface est manquante :")
    print(df.loc[df[surf].isna(), "type_batiment"].value_counts())
    print()

===== Département 59 =====
count    19695.0
mean        82.5
std        169.2
min          6.5
25%         46.9
50%         67.0
75%         89.2
max      10078.4
Name: surface_habitable_logement, dtype: float64
Surface manquante : 305
Surface < 9 m²    : 2
Surface > 300 m²  : 140
Surface > 500 m²  : 100
Hauteur sous plafond :
count    20000.00
mean         2.59
std          2.92
min          1.50
25%          2.50
50%          2.50
75%          2.50
max        310.00
Name: hauteur_sous_plafond, dtype: float64
Type de bâtiment, quand la surface est manquante :
type_batiment
immeuble       202
maison          64
appartement     39
Name: count, dtype: int64

===== Département 69 =====
count    19541.0
mean        77.9
std        141.9
min          8.0
25%         44.7
50%         64.0
75%         83.1
max       6317.5
Name: surface_habitable_logement, dtype: float64
Surface manquante : 459
Surface < 9 m²    : 3
Surface > 300 m²  : 167
Surface > 500 m²  : 125
Hauteur sous plafond :
count 

**Observations (étape 8)**
- Des surfaces impossibles : maximum de 10 078 m² (Lille), 6 317 m² (Lyon) et 101 036 m² (Paris). Environ 100 à 125 lignes par département dépassent 500 m².
- Les surfaces manquantes sont presque toutes des DPE de type `immeuble` (410 sur 412 à Paris). Un DPE d'immeuble décrit un bâtiment entier, pas un logement.
- La hauteur sous plafond est parfois saisie en centimètres au lieu de mètres (maximum de 310 à Lille et 270 à Lyon, alors que la médiane est de 2,5).
- Paris compte 90 logements de moins de 9 m² (contre 2 à Lille et 3 à Lyon) : probablement de vraies chambres de service, mais le minimum de 3,4 m² est suspect.

## 9. Vérifications avant de fixer les règles
On regarde si les très grandes surfaces sont des immeubles, comment se répartissent les petites surfaces, et combien de hauteurs sous plafond sont en centimètres.

In [10]:
surf = "surface_habitable_logement"

for d, df in dfs.items():
    print(f"===== Département {d} =====")

    # Surface par type de bâtiment
    print(df.groupby("type_batiment")[surf].agg(["count", "median", "max"]).round(1))

    # Type de bâtiment des surfaces > 300 m²
    print("Surfaces > 300 m², par type :", df.loc[df[surf] > 300, "type_batiment"].value_counts().to_dict())

    # Répartition des petites surfaces
    print("Petites surfaces :", pd.cut(df[surf], [0, 5, 7, 9]).value_counts().sort_index().to_dict())
    print()

===== Département 59 =====
               count  median      max
type_batiment                        
appartement    12105    52.4    592.4
immeuble         119  1217.1  10078.4
maison          7471    93.0   1800.4
Surfaces > 300 m², par type : {'immeuble': 105, 'maison': 30, 'appartement': 5}
Petites surfaces : {Interval(0, 5, closed='right'): 0, Interval(5, 7, closed='right'): 1, Interval(7, 9, closed='right'): 1}

===== Département 69 =====
               count  median     max
type_batiment                       
appartement    16377    60.6   362.4
immeuble         170   907.1  6317.5
maison          2994   110.0  1034.3
Surfaces > 300 m², par type : {'immeuble': 140, 'maison': 25, 'appartement': 2}
Petites surfaces : {Interval(0, 5, closed='right'): 0, Interval(5, 7, closed='right'): 0, Interval(7, 9, closed='right'): 4}

===== Département 75 =====
               count  median       max
type_batiment                         
appartement    18024    42.7    3931.9
immeuble       

**Observations (étape 9)**
- Les immeubles ont une surface médiane d'environ 1 000 m² (contre 40 à 110 m² pour les appartements et les maisons). Ils expliquent la plupart des surfaces supérieures à 300 m² et presque toutes les surfaces manquantes.
- Il reste quelques erreurs de saisie parmi les logements (par exemple un appartement de 3 932 m² à Paris).
- La hauteur sous plafond est parfois saisie en centimètres (270, 310) ou multipliée par 10 (25, 26). Il y a 119 valeurs supérieures à 5 m sur 58 644 lignes.
- Pour savoir si ces hauteurs concernent surtout les immeubles, on a comparé avant et après exclusion des immeubles (résultat ci-dessous).

**Décisions provisoires**
- Exclure les DPE de type `immeuble` (1 271 lignes, soit 2,2 %) : l'application concerne des logements, pas des bâtiments entiers.
- Supprimer les lignes sans surface.
- Surface minimale : 9 m². Surface maximale : à fixer (étape 10).
- Consommation maximale : 1 000 kWh/m²/an.

In [11]:
h = "hauteur_sous_plafond"

for d, df in dfs.items():
    print(f"===== Département {d} =====")
    print("Hauteur < 1.8 m       :", (df[h] < 1.8).sum())
    print("Hauteur entre 1.8 et 5 :", df[h].between(1.8, 5).sum())
    print("Hauteur > 5           :", (df[h] > 5).sum())
    print("Valeurs > 5 (extrait) :", sorted(df.loc[df[h] > 5, h].unique())[:15])
    print()

===== Département 59 =====
Hauteur < 1.8 m       : 1
Hauteur entre 1.8 et 5 : 19913
Hauteur > 5           : 86
Valeurs > 5 (extrait) : [np.float64(5.1), np.float64(5.2), np.float64(5.3), np.float64(5.4), np.float64(5.5), np.float64(5.7), np.float64(5.8), np.float64(5.9), np.float64(6.0), np.float64(6.1), np.float64(6.3), np.float64(6.5), np.float64(6.9), np.float64(7.1), np.float64(7.5)]

===== Département 69 =====
Hauteur < 1.8 m       : 7
Hauteur entre 1.8 et 5 : 19964
Hauteur > 5           : 29
Valeurs > 5 (extrait) : [np.float64(5.3), np.float64(5.8), np.float64(6.0), np.float64(7.2), np.float64(25.0), np.float64(25.5), np.float64(26.0), np.float64(270.0)]

===== Département 75 =====
Hauteur < 1.8 m       : 3
Hauteur entre 1.8 et 5 : 18637
Hauteur > 5           : 4
Valeurs > 5 (extrait) : [np.float64(5.5), np.float64(6.6), np.float64(9.5), np.float64(16.2)]



In [12]:
h = "hauteur_sous_plafond"

for d, df in dfs.items():
    hautes = df[df[h] > 5]
    print(f"===== Département {d} : {len(hautes)} valeurs > 5 m =====")
    print(hautes["type_batiment"].value_counts().to_dict())

    sans_immeuble = df[df["type_batiment"] != "immeuble"]
    print("Hors immeubles :", (sans_immeuble[h] > 5).sum(), "valeurs > 5 m")
    print()

===== Département 59 : 86 valeurs > 5 m =====
{'appartement': 49, 'maison': 36, 'immeuble': 1}
Hors immeubles : 85 valeurs > 5 m

===== Département 69 : 29 valeurs > 5 m =====
{'appartement': 24, 'maison': 3, 'immeuble': 2}
Hors immeubles : 27 valeurs > 5 m

===== Département 75 : 4 valeurs > 5 m =====
{'appartement': 2, 'immeuble': 1, 'maison': 1}
Hors immeubles : 3 valeurs > 5 m



**Résultat (hauteur sous plafond)**
- Sur 119 valeurs supérieures à 5 m, seulement 4 concernent des immeubles : 115 concernent des appartements et des maisons (85 à Lille, 27 à Lyon, 3 à Paris).
- Exclure les immeubles ne suffit donc pas pour corriger cette variable.

**Décision (hauteur sous plafond)** : les valeurs hors de 1,8 à 5 m sont remplacées par « manquant ». On ne supprime pas la ligne et on ne devine pas l'unité de saisie (mètres ou centimètres). Le modèle remplacera le manquant par une valeur typique (la médiane) au moment de l'entraînement.

## 10. Surface maximale et coordonnées
On regarde la répartition des grandes surfaces parmi les logements (hors immeubles), puis le format des coordonnées pour la carte.

In [13]:
surf = "surface_habitable_logement"

for d, df in dfs.items():
    logements = df[df["type_batiment"] != "immeuble"]
    print(f"===== Département {d} : {len(logements)} logements (hors immeubles) =====")
    print(pd.cut(logements[surf], [0, 9, 200, 300, 400, 500, 1000, 1e6]).value_counts().sort_index())
    print()

===== Département 59 : 19679 logements (hors immeubles) =====
surface_habitable_logement
(0.0, 9.0]                 2
(9.0, 200.0]           19352
(200.0, 300.0]           187
(300.0, 400.0]            23
(400.0, 500.0]             4
(500.0, 1000.0]            7
(1000.0, 1000000.0]        1
Name: count, dtype: int64

===== Département 69 : 19583 logements (hors immeubles) =====
surface_habitable_logement
(0.0, 9.0]                 4
(9.0, 200.0]           19163
(200.0, 300.0]           177
(300.0, 400.0]            19
(400.0, 500.0]             5
(500.0, 1000.0]            2
(1000.0, 1000000.0]        1
Name: count, dtype: int64

===== Département 75 : 18111 logements (hors immeubles) =====
surface_habitable_logement
(0.0, 9.0]               106
(9.0, 200.0]           17938
(200.0, 300.0]            44
(300.0, 400.0]            14
(400.0, 500.0]             1
(500.0, 1000.0]            4
(1000.0, 1000000.0]        2
Name: count, dtype: int64



In [14]:
for d, df in dfs.items():
    print(d, "-> coordonnées manquantes :", df["_geopoint"].isna().sum())

print()
print(dfs["69"]["_geopoint"].head())
print(type(dfs["69"]["_geopoint"].iloc[0]))

59 -> coordonnées manquantes : 0
69 -> coordonnées manquantes : 0
75 -> coordonnées manquantes : 0

0    46.001749036869164,4.710939968070691
1     45.77587496570502,4.918295955089466
2     45.74137898096509,4.861241989402495
3     45.74496000928658,4.863709027217594
4     45.77250803193388,4.880935964606717
Name: _geopoint, dtype: str
<class 'str'>


**Observations (étape 10)**
- Hors immeubles, plus de 97 % des logements ont une surface entre 9 et 200 m².
- De 200 à 500 m², ce sont de grandes maisons plausibles (environ 280 lignes au total).
- Au-dessus de 500 m², il ne reste que 17 lignes sur environ 57 000 (8 à Lille, 3 à Lyon, 6 à Paris) : erreurs de saisie probables.
- Paris compte 106 logements de 9 m² ou moins (contre 2 à Lille et 4 à Lyon).
- La colonne `_geopoint` est un texte au format « latitude,longitude », sans valeur manquante.

**Décisions (surface)**
- Surface minimale : 9 m². Surface maximale : 500 m².
- Les coordonnées sont extraites en découpant `_geopoint` au niveau de la virgule.
## 11. Coordonnées et neufs classés D, E ou G
On vérifie que les coordonnées sont convertibles et situées en France, puis on regarde les logements neufs avec une mauvaise étiquette.

In [16]:
for d, df in dfs.items():
    coords = df["_geopoint"].str.split(",", expand=True)
    lat = pd.to_numeric(coords[0], errors="coerce")
    lon = pd.to_numeric(coords[1], errors="coerce")

    print(f"===== Département {d} =====")
    print("Non convertibles :", lat.isna().sum())
    print("Latitude  :", lat.min().round(2), "->", lat.max().round(2))
    print("Longitude :", lon.min().round(2), "->", lon.max().round(2))
    print("Hors de la France métropolitaine :",
          (~(lat.between(41, 52) & lon.between(-5.5, 10))).sum())
    print()

===== Département 59 =====
Non convertibles : 0
Latitude  : -5.98 -> 51.08
Longitude : -1.36 -> 4.2
Hors de la France métropolitaine : 22

===== Département 69 =====
Non convertibles : 0
Latitude  : -5.98 -> 46.28
Longitude : -1.36 -> 5.13
Hors de la France métropolitaine : 184

===== Département 75 =====
Non convertibles : 0
Latitude  : 48.82 -> 48.9
Longitude : 2.25 -> 2.41
Hors de la France métropolitaine : 0



In [17]:
cols = ["etiquette_dpe", "etiquette_ges", "conso_5_usages_par_m2_ep",
        "emission_ges_5_usages_par_m2", "surface_habitable_logement",
        "type_energie_principale_chauffage"]

for d, df in dfs.items():
    neufs_ge = df[(df["origine"] == "dpe_neufs") & df["etiquette_dpe"].isin(["E", "G"])]
    print(f"===== Département {d} : {len(neufs_ge)} neufs en E ou G =====")
    if len(neufs_ge):
        print(neufs_ge[cols].sort_values("etiquette_dpe"))
    print()

===== Département 59 : 1 neufs en E ou G =====
      etiquette_dpe etiquette_ges  conso_5_usages_par_m2_ep  \
17650             G             G                     639.0   

       emission_ges_5_usages_par_m2  surface_habitable_logement  \
17650                         119.6                        73.7   

      type_energie_principale_chauffage  
17650                       Gaz naturel  

===== Département 69 : 14 neufs en E ou G =====
      etiquette_dpe etiquette_ges  conso_5_usages_par_m2_ep  \
14204             E             C                     306.1   
15230             E             B                     264.1   
13732             G             G                      60.1   
14072             G             G                      61.6   
14309             G             G                      60.1   
16192             G             G                      60.9   
16830             G             G                      60.4   
16832             G             G                     

**Observations (étape 11)**
- Coordonnées : 22 points invalides à Lille et 184 à Lyon, tous situés à la même position (latitude -5,98, longitude -1,36), en plein océan. Probablement une valeur par défaut après un échec de géocodage. Paris est cohérent.
- Neufs en G à Lyon (12 lignes) : consommation de 58 à 83 kWh/m²/an (niveau A ou B) mais émissions de GES de 5 900 à 8 600 kg CO₂/m²/an, ce qui est impossible. L'étiquette G vient donc d'une erreur de saisie des émissions. Tous sont chauffés par un réseau de chauffage urbain.
- Neuf en G à Lille (1 ligne) et neufs en E (3 lignes) : étiquettes cohérentes avec la consommation, on les garde.

**Décisions (étape 11)**
- Coordonnées hors de la France métropolitaine : latitude et longitude remplacées par « manquant » (la ligne est conservée).
- Émissions de GES aberrantes : lignes supprimées (seuil à fixer).

## 12. Vérifications : coordonnées invalides et émissions de GES
On vérifie que les coordonnées invalides sont bien un échec de géocodage, puis on fixe le seuil des émissions de GES aberrantes.

In [18]:
for d, df in dfs.items():
    coords = df["_geopoint"].str.split(",", expand=True)
    lat = pd.to_numeric(coords[0], errors="coerce")
    lon = pd.to_numeric(coords[1], errors="coerce")
    invalide = ~(lat.between(41, 52) & lon.between(-5.5, 10))

    print(f"===== Département {d} : {invalide.sum()} coordonnées invalides =====")
    print(df.loc[invalide, "_geopoint"].value_counts().head(3))
    print("Statut de géocodage :", df.loc[invalide, "statut_geocodage"].value_counts(dropna=False).to_dict())
    print("Type de logement    :", df.loc[invalide, "origine"].value_counts().to_dict())
    print()

===== Département 59 : 22 coordonnées invalides =====
_geopoint
-5.98385630920877,-1.363081210117898    22
Name: count, dtype: int64
Statut de géocodage : {"adresse géocodée ban à l'adresse": 22}
Type de logement    : {'dpe_existants': 22}

===== Département 69 : 184 coordonnées invalides =====
_geopoint
-5.98385630920877,-1.363081210117898     183
-5.983569780356801,-1.363065199959244      1
Name: count, dtype: int64
Statut de géocodage : {"adresse géocodée ban à l'adresse": 184}
Type de logement    : {'dpe_existants': 184}

===== Département 75 : 0 coordonnées invalides =====
Series([], Name: count, dtype: int64)
Statut de géocodage : {}
Type de logement    : {}



In [19]:
e = "emission_ges_5_usages_par_m2"

for d, df in dfs.items():
    print(f"===== Département {d} =====")
    print(df[e].quantile([0.5, 0.99, 0.999, 1]).round(1).to_dict())
    print("Émissions > 200 :", (df[e] > 200).sum())
    print(df[df[e] > 200].groupby(["origine", "etiquette_dpe"]).size())
    print()

===== Département 59 =====
{0.5: 13.0, 0.99: 91.0, 0.999: 157.0, 1.0: 456.0}
Émissions > 200 : 8
origine        etiquette_dpe
dpe_existants  G                8
dtype: int64

===== Département 69 =====
{0.5: 12.7, 0.99: 86.0, 0.999: 173.0, 1.0: 8570.9}
Émissions > 200 : 16
origine        etiquette_dpe
dpe_existants  G                 4
dpe_neufs      G                12
dtype: int64

===== Département 75 =====
{0.5: 13.0, 0.99: 94.0, 0.999: 155.2, 1.0: 320.3}
Émissions > 200 : 2
origine        etiquette_dpe
dpe_existants  G                2
dtype: int64

